In [1]:
# Category: Clothing, Shoes & Jewelry
# ==============================================================================

# ── 1. CÀI ĐẶT & IMPORT ────────────────────────────────────────────────────────
!pip install huggingface_hub pyarrow scikit-learn -q

import os
import re
import json
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from pathlib import Path
from huggingface_hub import snapshot_download
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# ── 2. DOWNLOAD DỮ LIỆU CHỈ ĐỊNH (Không sợ tràn đĩa) ───────────────────────────
REPO_ID   = "datdong2004/amazonNew-cleaned"
CATEGORY  = "Clothing_Shoes_and_Jewelry"
HF_TOKEN  = None  # Điền token bắt đầu bằng hf_... vào đây nếu bị giới hạn tải

LOCAL_DIR = f"/kaggle/working/data/{CATEGORY}"
os.makedirs(LOCAL_DIR, exist_ok=True)

print(f"1. Đang tải dữ liệu {CATEGORY}...")
snapshot_download(
    repo_id         = REPO_ID,
    repo_type       = "dataset",
    local_dir       = LOCAL_DIR,
    token           = HF_TOKEN,
    # Sửa bộ lọc: Tìm kiếm linh hoạt hơn để không bị sót file
    allow_patterns  = ["*Clothing*", "*clothing*"], 
    ignore_patterns = ["*.json", "*.md", "*.txt"],
)

downloaded = []
for root, dirs, fnames in os.walk(LOCAL_DIR):
    for f in fnames:
        if f.endswith(".parquet"):
            downloaded.append(os.path.join(root, f))
print(f"-> Đã tải xong {len(downloaded)} files Parquet.\n")

if len(downloaded) == 0:
    raise ValueError("Không tìm thấy file nào! Hãy kiểm tra lại REPO_ID hoặc Hugging Face đang bảo trì.")

# ── 3. LOAD DATA & PARSE PARTITIONS ────────────────────────────────────────────
print("2. Gộp dữ liệu thành DataFrame...")
COL_USER   = "reviewerID"
COL_ITEM   = "asin"
COL_RATING = "overall"
COL_DATE   = "review_date"
NEEDED     = [COL_USER, COL_ITEM, COL_DATE] 

dfs = []
for path in downloaded:
    # Lấy số sao (rating) từ tên thư mục (ví dụ: overall=5)
    match = re.search(r'overall=(\d)', path)
    overall_val = int(match.group(1)) if match else None
    
    pf = pq.ParquetFile(path)
    df_chunk = pf.read(columns=NEEDED).to_pandas()
    df_chunk[COL_RATING] = overall_val
    dfs.append(df_chunk)

df = pd.concat(dfs, ignore_index=True)
df[COL_DATE]   = pd.to_datetime(df[COL_DATE])
df[COL_RATING] = df[COL_RATING].astype(int)

print(f"-> Tổng số reviews ban đầu: {len(df):,}\n")

# ── 4. K-CORE FILTERING (K=5) ──────────────────────────────────────────────────
K = 5
print(f"3. Lọc {K}-core...")
df_core = df.copy()
iteration = 0
while True:
    iteration += 1
    user_counts = df_core[COL_USER].value_counts()
    item_counts = df_core[COL_ITEM].value_counts()
    
    valid_users = user_counts[user_counts >= K].index
    valid_items = item_counts[item_counts >= K].index
    
    df_filtered = df_core[
        df_core[COL_USER].isin(valid_users) &
        df_core[COL_ITEM].isin(valid_items)
    ]
    
    n_removed = len(df_core) - len(df_filtered)
    if n_removed == 0:
        break
    df_core = df_filtered

print(f"-> Sau lọc: {len(df_core):,} reviews | {df_core[COL_USER].nunique():,} users | {df_core[COL_ITEM].nunique():,} items\n")

# ── 5. TEMPORAL SPLIT 80/20 & PHÂN LOẠI TEST ──────────────────────────────────
print("4. Chia dữ liệu (Temporal Split)...")
df_core = df_core.sort_values(COL_DATE).reset_index(drop=True)
split_idx  = int(len(df_core) * 0.8)
split_date = df_core.iloc[split_idx][COL_DATE]

train = df_core.iloc[:split_idx].copy()
test  = df_core.iloc[split_idx:].copy()

train_users = set(train[COL_USER].unique())
train_items = set(train[COL_ITEM].unique())

test["user_known"] = test[COL_USER].isin(train_users)
test["item_known"] = test[COL_ITEM].isin(train_items)
mask_warm    = test["user_known"] & test["item_known"]
mask_cold_u  = ~test["user_known"] & test["item_known"]
mask_cold_i  = test["user_known"] & ~test["item_known"]
mask_cold_ui = ~test["user_known"] & ~test["item_known"]

print(f"-> Train: {len(train):,} | Test: {len(test):,}")
print(f"-> Test Warm: {mask_warm.sum():,} | Cold-U: {mask_cold_u.sum():,} | Cold-I: {mask_cold_i.sum():,} | Fully-Cold: {mask_cold_ui.sum():,}\n")

# ── 6. BAYESIAN SHRINKAGE BASELINE ─────────────────────────────────────────────
print("5. Chạy Baseline & Đánh giá...")
global_mean = train[COL_RATING].mean()
user_stats = train.groupby(COL_USER)[COL_RATING].agg(["mean","count"]).rename(columns={"mean": "user_mean_raw", "count": "user_n"})
item_stats = train.groupby(COL_ITEM)[COL_RATING].agg(["mean","count"]).rename(columns={"mean": "item_mean_raw", "count": "item_n"})

m_user = float(user_stats["user_n"].median())
m_item = float(item_stats["item_n"].median())

user_stats["user_mean"] = (user_stats["user_n"] * user_stats["user_mean_raw"] + m_user * global_mean) / (user_stats["user_n"] + m_user)
item_stats["item_mean"] = (item_stats["item_n"] * item_stats["item_mean_raw"] + m_item * global_mean) / (item_stats["item_n"] + m_item)

user_mean_dict = user_stats["user_mean"].to_dict()
item_mean_dict = item_stats["item_mean"].to_dict()

def predict_vectorized(df_subset):
    u = df_subset[COL_USER].map(user_mean_dict).fillna(global_mean)
    i = df_subset[COL_ITEM].map(item_mean_dict).fillna(global_mean)
    return ((u + i) / 2.0).clip(1.0, 5.0)

test["pred"]   = predict_vectorized(test)
test["actual"] = test[COL_RATING]

warm_rmse = mean_squared_error(test.loc[mask_warm, "actual"], test.loc[mask_warm, "pred"]) ** 0.5
warm_mae  = mean_absolute_error(test.loc[mask_warm, "actual"], test.loc[mask_warm, "pred"])
print(f"-> WARM TEST RMSE: {warm_rmse:.4f} | MAE: {warm_mae:.4f}\n")

# ── 7. LƯU ARTIFACTS CHO STAGE SAU ─────────────────────────────────────────────
print("6. Đang lưu files...")
ARTIFACTS_DIR = Path("/kaggle/working/stage0_artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Lọc cột và gắn cờ cho tập test
train_to_save = train[[COL_USER, COL_ITEM, COL_RATING, COL_DATE]].copy()
test_to_save = test[[COL_USER, COL_ITEM, COL_RATING, COL_DATE, "user_known", "item_known"]].copy()
test_to_save["is_warm"] = mask_warm.values
test_to_save["is_cold_user"] = mask_cold_u.values
test_to_save["is_cold_item"] = mask_cold_i.values
test_to_save["is_cold_both"] = mask_cold_ui.values

# Lưu file dạng parquet nén zstd cực nhẹ
train_to_save.to_parquet(ARTIFACTS_DIR / "train.parquet", compression="zstd", index=False)
test_to_save.to_parquet(ARTIFACTS_DIR / "test.parquet", compression="zstd", index=False)
df_core[[COL_USER, COL_ITEM, COL_RATING, COL_DATE]].to_parquet(ARTIFACTS_DIR / "df_core.parquet", compression="zstd", index=False)
user_stats[["user_mean","user_mean_raw","user_n"]].to_parquet(ARTIFACTS_DIR / "user_means.parquet", compression="zstd")
item_stats[["item_mean","item_mean_raw","item_n"]].to_parquet(ARTIFACTS_DIR / "item_means.parquet", compression="zstd")

print(f"-> Đã lưu tất cả thành công vào: {ARTIFACTS_DIR}")
print("=== HOÀN THÀNH STAGE 0 ===")

1. Đang tải dữ liệu Clothing_Shoes_and_Jewelry...


Fetching ... files: 0it [00:00, ?it/s]

-> Đã tải xong 80 files Parquet.

2. Gộp dữ liệu thành DataFrame...
-> Tổng số reviews ban đầu: 17,714,600

3. Lọc 5-core...
-> Sau lọc: 8,082,980 reviews | 929,819 users | 326,704 items

4. Chia dữ liệu (Temporal Split)...
-> Train: 6,466,384 | Test: 1,616,596
-> Test Warm: 1,349,731 | Cold-U: 234,224 | Cold-I: 28,556 | Fully-Cold: 4,085

5. Chạy Baseline & Đánh giá...
-> WARM TEST RMSE: 1.1288 | MAE: 0.8800

6. Đang lưu files...
-> Đã lưu tất cả thành công vào: /kaggle/working/stage0_artifacts
=== HOÀN THÀNH STAGE 0 ===


In [2]:
# ================================================================
# STAGE 2 — FEATURE ENGINEERING (VECTORIZED — ước tính ~15-30 phút)
# Category: Clothing, Shoes & Jewelry
# ================================================================

# ── 1. IMPORT & SETUP ────────────────────────────────────────────
import os, gc, pickle, re, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import pyarrow.parquet as pq
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from collections import Counter
warnings.filterwarnings('ignore')

CATEGORY      = "Clothing_Shoes_and_Jewelry"
LOCAL_DIR     = f"/kaggle/working/data/{CATEGORY}"
STAGE0_DIR    = Path("/kaggle/working/stage0_artifacts")
ARTIFACTS_DIR = Path("/kaggle/working/stage2_artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

COL_USER   = "reviewerID"
COL_ITEM   = "asin"
COL_RATING = "overall"
COL_DATE   = "review_date"

# ── 2. LOAD TRAIN ────────────────────────────────────────────────
print("1. Load train từ Stage 0...")
train = pd.read_parquet(STAGE0_DIR / "train.parquet")

global_mean    = train[COL_RATING].mean()
train_user_ids = set(train[COL_USER].unique())
train_item_ids = set(train[COL_ITEM].unique())

# Chuyển timestamp một lần duy nhất — tránh lặp lại sau này
train["timestamp"] = pd.to_datetime(train[COL_DATE]).astype("int64") // 10**9
max_ts = int(train["timestamp"].max())

all_items = sorted(train_item_ids)
all_users = sorted(train_user_ids)
n_items   = len(all_items)
n_users   = len(all_users)
item2idx  = {iid: i for i, iid in enumerate(all_items)}
user2idx  = {uid: i for i, uid in enumerate(all_users)}

# Map sang index ngay trên train — dùng được cho mọi bước sau
train["u_idx"] = train[COL_USER].map(user2idx)
train["i_idx"] = train[COL_ITEM].map(item2idx)

print(f"   Train: {len(train):,} | Users: {n_users:,} | Items: {n_items:,}")
print(f"   Global mean: {global_mean:.4f}\n")

# ── 3. LOAD TEXT/META (chỉ cột cần, chỉ items trong train) ───────
print("2. Load text/meta từ Parquet files...")
downloaded = sorted([
    os.path.join(r, f)
    for r, _, files in os.walk(LOCAL_DIR)
    for f in files if f.endswith(".parquet")
])
print(f"   {len(downloaded)} file(s) tìm thấy.")

WANT = [COL_USER, COL_ITEM, "reviewText", "summary", "title", "brand", "price"]
dfs  = []
for path in downloaded:
    pf   = pq.ParquetFile(path)
    cols = [c for c in WANT if c in pf.schema.names]
    chunk = pf.read(columns=cols).to_pandas()
    # Chỉ giữ items có trong train — không cần filter theo user
    chunk = chunk[chunk[COL_ITEM].isin(train_item_ids)]
    if len(chunk):
        dfs.append(chunk)

df_meta = pd.concat(dfs, ignore_index=True)
del dfs; gc.collect()
print(f"   df_meta: {len(df_meta):,} rows | cols: {df_meta.columns.tolist()}\n")

# ── 4. BUILD ITEM TEXT CORPUS (vectorized groupby) ───────────────
print("3. Build item text corpus...")

text_cols = [c for c in ["reviewText", "summary", "title"] if c in df_meta.columns]
if text_cols:
    # Gom tất cả text cols thành 1 cột, xử lý trùng tên nếu có
    def safe_series(df, col):
        s = df[col]
        return s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s

    df_meta["_text"] = safe_series(df_meta, text_cols[0]).fillna("")
    for col in text_cols[1:]:
        df_meta["_text"] = df_meta["_text"] + " " + safe_series(df_meta, col).fillna("")
else:
    df_meta["_text"] = ""

# groupby + join — nhanh hơn iterrows hàng chục lần
item_text_series = (
    df_meta.groupby(COL_ITEM)["_text"]
    .apply(" ".join)
)
item_text_list = [item_text_series.get(iid, "") for iid in all_items]
print(f"   Items có text: {sum(1 for t in item_text_list if t.strip()):,} / {n_items:,}")

# ── 5. TF-IDF (3000 features) ────────────────────────────────────
print("\n4. TF-IDF review text (3000 features)...")
tfidf_review = TfidfVectorizer(
    max_features=3000, min_df=3, max_df=0.95,
    strip_accents="unicode", sublinear_tf=True,
)
M_review = tfidf_review.fit_transform(item_text_list)
print(f"   M_review: {M_review.shape}")
del item_text_list; gc.collect()

# ── 6. NUMERIC FEATURES (vectorized) ─────────────────────────────
print("\n5. Numeric features...")
item_stats = train.groupby(COL_ITEM)[COL_RATING].agg(["mean", "count", "var"]).fillna(0)

# Price
price_map  = {}
if "price" in df_meta.columns:
    price_raw = (df_meta.groupby(COL_ITEM)["price"]
                         .first()
                         .astype(str)
                         .str.replace(r"[^0-9.]", "", regex=True))
    price_map = pd.to_numeric(price_raw, errors="coerce").dropna().to_dict()
price_mean = float(np.mean(list(price_map.values()))) if price_map else 20.0

# Vectorized build — không dùng vòng for
avg_r_arr  = np.array([item_stats.loc[iid, "mean"]  if iid in item_stats.index else global_mean for iid in all_items], dtype=np.float32)
n_r_arr    = np.array([item_stats.loc[iid, "count"] if iid in item_stats.index else 0           for iid in all_items], dtype=np.float32)
var_r_arr  = np.array([item_stats.loc[iid, "var"]   if iid in item_stats.index else 0           for iid in all_items], dtype=np.float32)
price_arr  = np.array([price_map.get(iid, price_mean) for iid in all_items], dtype=np.float32)
price_arr  = np.nan_to_num(price_arr, nan=price_mean)

numeric_matrix = np.column_stack([
    (avg_r_arr - 1) / 4.0,
    np.log1p(n_r_arr) / 10.0,
    np.minimum(var_r_arr / 4.0, 1.0),
    np.minimum(price_arr / 200.0, 2.0),
]).astype(np.float32)
M_numeric = sp.csr_matrix(numeric_matrix)
print(f"   M_numeric: {M_numeric.shape}")

# ── 7. ONE-HOT BRAND (top 100) ────────────────────────────────────
print("\n6. One-hot brand...")
brand_map = {}
if "brand" in df_meta.columns:
    brand_map = (df_meta.groupby(COL_ITEM)["brand"]
                         .first()
                         .fillna("unknown")
                         .str.lower().str.strip()
                         .to_dict())

all_brands = [brand_map.get(iid, "unknown") for iid in all_items]
top_brands = [b for b, _ in Counter(all_brands).most_common(101) if b != "unknown"][:100]
b2idx      = {b: i for i, b in enumerate(top_brands)}

rows_b = [i for i, iid in enumerate(all_items) if brand_map.get(iid, "unknown") in b2idx]
cols_b = [b2idx[brand_map.get(all_items[i], "unknown")] for i in rows_b]
M_brand = sp.csr_matrix(
    (np.ones(len(rows_b), dtype=np.float32), (rows_b, cols_b)),
    shape=(n_items, max(len(top_brands), 1))
)
print(f"   M_brand: {M_brand.shape}")
del df_meta; gc.collect()

# ── 8. SVD 128 ───────────────────────────────────────────────────
print("\n7. Concat + TruncatedSVD (128 components)...")
M_raw = sp.hstack([M_review, M_numeric, M_brand]).tocsr()
print(f"   M_raw: {M_raw.shape}")

svd = TruncatedSVD(n_components=128, random_state=42, n_iter=5)
item_profiles = svd.fit_transform(M_raw).astype(np.float32)
print(f"   item_profiles: {item_profiles.shape}")
print(f"   Explained var: {svd.explained_variance_ratio_.sum()*100:.2f}%")
del M_raw, M_review, M_numeric, M_brand; gc.collect()

# ── 9. USER PROFILES — VECTORIZED (đây là điểm mấu chốt) ─────────
print("\n8. Build user profiles (VECTORIZED)...")
# ❌ Cách cũ: for uid in users: for _, row in grp.iterrows() → O(N) Python loops = 13h
# ✅ Cách mới: sparse matrix multiply → O(1) với numpy/scipy = vài phút

LAMBDA = 0.001

# Bước 1: Tính time-decay weight cho mỗi review (vectorized)
days_old   = (max_ts - train["timestamp"].values) / 86400.0
time_w     = np.exp(-LAMBDA * days_old).astype(np.float32)

# Bước 2: Tính rating-deviation weight (vectorized)
user_mean_mapped = train[COL_USER].map(
    train.groupby(COL_USER)[COL_RATING].mean()
).values.astype(np.float32)
rating_dev_w = (np.abs(train[COL_RATING].values - user_mean_mapped) + 0.1).astype(np.float32)

# Bước 3: Kết hợp thành final weight
final_w = time_w * rating_dev_w   # shape: (n_train_rows,)

# Bước 4: Build sparse weight matrix W (n_users × n_train_rows)
#         rồi nhân với item_profiles — hoàn toàn vectorized
u_idx_arr = train["u_idx"].values.astype(np.int32)
i_idx_arr = train["i_idx"].values.astype(np.int32)

# Ma trận trọng số: W[u, k] = weight của review thứ k nếu thuộc user u
W_sparse = sp.csr_matrix(
    (final_w, (u_idx_arr, np.arange(len(train)))),
    shape=(n_users, len(train))
)

# item_profiles tại từng review: shape (n_train_rows, 128)
item_vecs_per_review = item_profiles[i_idx_arr]   # numpy fancy indexing

# user_profiles_raw[u] = sum of w_k * item_profiles[i_k] for reviews of user u
user_profiles_raw = W_sparse.dot(item_vecs_per_review)   # (n_users, 128)

# Bước 5: Normalize theo tổng weight của từng user
weight_sum = np.array(W_sparse.sum(axis=1)).flatten() + 1e-9   # (n_users,)
user_profiles = (user_profiles_raw / weight_sum[:, None]).astype(np.float32)

print(f"   user_profiles: {user_profiles.shape}")
del W_sparse, user_profiles_raw, item_vecs_per_review, final_w; gc.collect()

# ── 10. UTILITY + IMPLICIT + M_NORM ──────────────────────────────
print("\n9. Build sparse matrices...")
r_arr = train[COL_RATING].values.astype(np.float32)

utility_matrix  = sp.csr_matrix(
    (r_arr, (u_idx_arr, i_idx_arr)),
    shape=(n_users, n_items)
)
implicit_matrix = (utility_matrix > 0).astype(np.float32).tocsr()

# M_norm: trừ user mean (vectorized với sparse diags)
um_vec  = train.groupby(COL_USER)[COL_RATING].mean().reindex(all_users, fill_value=global_mean).values.astype(np.float32)
ind_mat = implicit_matrix.copy()
M_norm  = (utility_matrix - sp.diags(um_vec).dot(ind_mat)).tocsr()

sparsity = 1 - utility_matrix.nnz / (n_users * n_items)
print(f"   utility : {utility_matrix.shape}  nnz={utility_matrix.nnz:,}")
print(f"   implicit: {implicit_matrix.shape}  nnz={implicit_matrix.nnz:,}")
print(f"   M_norm  : {M_norm.shape}  nnz={M_norm.nnz:,}")
print(f"   Sparsity: {sparsity*100:.4f}%")

# ── 11. SAVE ──────────────────────────────────────────────────────
print("\n10. Lưu artifacts...")
np.save(ARTIFACTS_DIR / "item_profiles.npy",  item_profiles)
np.save(ARTIFACTS_DIR / "user_profiles.npy",  user_profiles)
sp.save_npz(str(ARTIFACTS_DIR / "utility_matrix.npz"),  utility_matrix)
sp.save_npz(str(ARTIFACTS_DIR / "M_norm.npz"),           M_norm)
sp.save_npz(str(ARTIFACTS_DIR / "implicit_matrix.npz"),  implicit_matrix)

with open(ARTIFACTS_DIR / "stage2_artifacts.pkl", "wb") as f:
    pickle.dump({
        "item2idx"    : item2idx,
        "user2idx"    : user2idx,
        "all_items"   : all_items,
        "all_users"   : all_users,
        "svd"         : svd,
        "tfidf_review": tfidf_review,
        "top_brands"  : top_brands,
        "b2idx"       : b2idx,
        "um_vec"      : um_vec,
        "global_mean" : global_mean,
        "COL_USER"    : COL_USER,
        "COL_ITEM"    : COL_ITEM,
        "COL_RATING"  : COL_RATING,
    }, f)

print(f"   Saved to: {ARTIFACTS_DIR}")
for p in sorted(ARTIFACTS_DIR.iterdir()):
    print(f"   {p.name:<35} {p.stat().st_size/1e6:6.1f} MB")

print("\n=== HOÀN THÀNH STAGE 2 ===")

1. Load train từ Stage 0...
   Train: 6,466,384 | Users: 894,092 | Items: 323,629
   Global mean: 4.2463

2. Load text/meta từ Parquet files...
   80 file(s) tìm thấy.
   df_meta: 15,998,384 rows | cols: ['reviewerID', 'asin', 'reviewText', 'summary', 'title', 'brand', 'price']

3. Build item text corpus...
   Items có text: 323,629 / 323,629

4. TF-IDF review text (3000 features)...
   M_review: (323629, 3000)

5. Numeric features...
   M_numeric: (323629, 4)

6. One-hot brand...
   M_brand: (323629, 100)

7. Concat + TruncatedSVD (128 components)...
   M_raw: (323629, 3104)
   item_profiles: (323629, 128)
   Explained var: 49.69%

8. Build user profiles (VECTORIZED)...
   user_profiles: (894092, 128)

9. Build sparse matrices...
   utility : (894092, 323629)  nnz=6,448,442
   implicit: (894092, 323629)  nnz=6,448,442
   M_norm  : (894092, 323629)  nnz=5,544,228
   Sparsity: 99.9978%

10. Lưu artifacts...
   Saved to: /kaggle/working/stage2_artifacts
   M_norm.npz                     